# 7 - RERANKERS ON THE VAL SPLIT: the apples-to-apples comparison

Notebook 6's decision rule compared fine-tuned MiniLM at **81.1% R@5 on 1,626 val questions**
against v2-m3 at **85.0% on the 100-question sample**, and concluded "3.9% short". That
comparison is invalid: the 100-question sample is measurably easier (bi-encoder R@5 was
75.0% there vs 69.8% corpus-wide). The rule may have the answer backwards.

This runs every candidate on the **same 1,626 val questions**, so the decision rests on one
table instead of a transferred margin.

| candidate | why it is here |
|---|---|
| bi-encoder (pool order) | the no-reranker baseline; free, it is just position in the top-100 |
| MiniLM-L6 zero-shot | what fine-tuning started from (val R@5 69.1% per notebook 6) |
| **MiniLM-L6 fine-tuned** | notebook 6's output, val R@5 81.1% - should reproduce exactly |
| **bge-reranker-v2-m3** | the strongest off-the-shelf model, never measured on val |

**INFERENCE to be tested:** zero-shot MiniLM scored 69.1% on val and the corpus-wide
bi-encoder R@5 is 69.8%, i.e. MiniLM zero-shot ~= no reranker. On the easy sample v2-m3 beat
the bi-encoder by +10 points. If a similar margin holds here, v2-m3 lands near 79% and
**fine-tuned MiniLM (81.1%) wins while being 18x faster**. That is a prediction, recorded
before the run, not a result.

Two reproduction checks: MiniLM zero-shot must land on notebook 6's 69.1%, and fine-tuned MiniLM on
its 81.1%. If either drifts, the checkpoint or the data path is wrong and nothing below
should be trusted.

Cost: 163k pairs per model. ESTIMATE ~2 min each for the MiniLMs, ~36 min for v2-m3.

In [ ]:
# §0 SETUP
import os, sys, json, pickle, time, gc
sys.path.insert(0, os.getcwd())
import numpy as np
import torch
import nqrun
from sentence_transformers import CrossEncoder

In [ ]:
# §1 CONFIG
FT_LOCAL = '/workspace/artifacts/step4-rerank/minilm-ft'   # notebook 6's checkpoint, if on this pod
FT_HF    = 'step4-rerank/minilm-ft'                        # else pulled from HF
MODELS = [
    ('minilm-zeroshot', 'cross-encoder/ms-marco-MiniLM-L-6-v2'),
    ('minilm-finetuned', None),                            # resolved in §3
    ('bge-v2-m3', 'BAAI/bge-reranker-v2-m3'),
]
KS      = (1, 3, 5, 10, 20, 50, 100)
BATCH   = 64
MAXLEN  = 320
GROUP   = 10          # questions per predict call -> 1,000 pairs
VAL_SUB = None        # None = all 1,626; set an int for a faster loop

# Notebook 6, same val questions - these must reproduce
REF_NB6 = {'minilm-zeroshot': {1: 0.365, 3: 0.597, 5: 0.691, 10: 0.798},
          'minilm-finetuned': {1: 0.475, 3: 0.729, 5: 0.811, 10: 0.886}}

CHUNK, OVERLAP, STRIDE = 200, 50, 150
MAXT = 5
N_PAGES_EXPECTED, N_QS_EXPECTED, N_CHUNKS_EXPECTED = 18997, 16229, 469387
OUT, HF_DIR = nqrun.art('step4-rerank'), 'step2-retrieval'
print(f'{len(MODELS)} candidates on the val split | top-100 pools')
print(f'root {nqrun.ROOT} | run {nqrun.RUN_ID} | {nqrun.gpu_info()}')

## Definitions

Nothing below executes; running these cells only defines names.

In [ ]:
# DEFINITIONS — derivation identical to notebook 6, so the two are directly comparable.
# chunks_for_span reads the module-level `base` and `n_per_page`.
def chunks_for_span(p, lo, hi):
    """Global chunk ids on page p that wholly contain the word span [lo, hi)."""
    j0 = max(0, -(-(hi - CHUNK) // STRIDE))
    j1 = min(lo // STRIDE, n_per_page[p] - 1)
    return [int(base[p] + j) for j in range(j0, j1 + 1)]

def resolve_ft():
    """Notebook 6's checkpoint: local if this is the same pod, else pulled from HF."""
    if os.path.isdir(FT_LOCAL) and os.path.exists(os.path.join(FT_LOCAL, 'config.json')):
        print(f'  fine-tuned checkpoint: local {FT_LOCAL}')
        return FT_LOCAL
    from huggingface_hub import snapshot_download
    p = snapshot_download(repo_id=nqrun.HF_REPO, repo_type='dataset',
                          allow_patterns=f'{FT_HF}/*')
    path = os.path.join(p, FT_HF)
    print(f'  fine-tuned checkpoint: pulled from HF -> {path}')
    return path

# §4 REPORT
def rec(r):
    """Recall@k for an array of gold ranks."""
    return {k: float((r < k).mean()) for k in KS}

def mrr(r):
    """Mean reciprocal rank, scoring unreachable questions as 0."""
    return float(np.where(r < UNREACHABLE, 1.0 / np.minimum(r + 1, 10 ** 6), 0.0).mean())

## Orchestration

In [ ]:
# §2 DATA + GUARDS - identical derivation to notebook 6 so the comparison is exact.
IDX = json.load(open(nqrun.hf_pull(f'{HF_DIR}/chunk_index.json')))
H, owner = IDX['doc_hashes'], np.array(IDX['owner'], dtype='int32')
assert IDX['chunk'] == CHUNK and IDX['overlap'] == OVERLAP
assert len(H) == N_PAGES_EXPECTED and len(owner) == N_CHUNKS_EXPECTED
n_per_page = np.bincount(owner, minlength=len(H)).astype('int64')
base = np.concatenate([[0], np.cumsum(n_per_page)[:-1]]).astype('int64')
top100 = np.load(nqrun.hf_pull(f'{HF_DIR}/top100_chunks.npy'))

SP = json.load(open(nqrun.hf_pull('step4-rerank/split.json')))
val_idx = np.array(SP['val_idx'])
assert SP['n_questions'] == N_QS_EXPECTED, 'split built on a different corpus'
qtype = np.array(SP['question_type'])

C = pickle.load(open(nqrun.hf_pull('step1-clean/clean_corpus.pkl'), 'rb'))
docs, questions = C['documents'], C['questions']
assert len(docs) == N_PAGES_EXPECTED and len(questions) == N_QS_EXPECTED, 'WRONG ARTIFACT'
assert list(docs) == H
row = {h: i for i, h in enumerate(H)}

texts = []
for h in H:
    w, title = docs[h]['words'], docs[h]['title']
    for a in range(0, max(len(w), 1), STRIDE):
        part = w[a:a + CHUNK]
        if not part:
            break
        texts.append(f"{title}. {' '.join(part)}")
assert len(texts) == N_CHUNKS_EXPECTED

gold_sets = [{c for _, lo, hi in q['gold']
              for c in chunks_for_span(row[q['doc_hash']], lo, hi)} for q in questions]
qtext = [q['question'] for q in questions]

bad = 0
for i in np.random.default_rng(0).choice(len(questions), 300, replace=False):
    q = questions[i]
    for k, (_, lo, hi) in enumerate(q['gold']):
        for c in chunks_for_span(row[q['doc_hash']], lo, hi):
            if q['answers_rendered'][k] not in texts[c] or owner[c] != row[q['doc_hash']]:
                bad += 1
print(f'alignment check (300 sampled): {bad} mismatches  <- must be 0')
assert bad == 0
del C, docs; gc.collect()

ev = val_idx if VAL_SUB is None else np.random.default_rng(0).choice(
    val_idx, VAL_SUB, replace=False)
UNREACHABLE = 10 ** 9

# bi-encoder baseline: the pool is already in bge rank order
bi = np.full(len(ev), UNREACHABLE, dtype='int64')
gold_pos = []
for n, i in enumerate(ev):
    hit = [j for j, c in enumerate(top100[i].tolist()) if c in gold_sets[i]]
    gold_pos.append(hit)
    if hit:
        bi[n] = min(hit)
CEIL = float((bi < 100).mean())
print(f'val questions: {len(ev):,} | ceiling (gold in the top-100 pool): {CEIL:.1%}')
print(f'bi-encoder (no reranker): ' + '  '.join(f'R@{k} {(bi < k).mean():.1%}' for k in KS))

In [ ]:
# §3 RUN each candidate. Failures are per-model so one bad load cannot cost the run.

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
ranks, rate, failed = {}, {}, {}

for name, mid in MODELS:
    try:
        src = resolve_ft() if mid is None else mid
        ce = CrossEncoder(src, max_length=MAXLEN, device=dev)
        r = np.full(len(ev), UNREACHABLE, dtype='int64')
        t0 = time.time()
        for a in range(0, len(ev), GROUP):
            ns = list(range(a, min(a + GROUP, len(ev))))
            pairs = [[qtext[ev[n]], texts[int(c)]] for n in ns for c in top100[ev[n]]]
            s = np.asarray(ce.predict(pairs, batch_size=BATCH, show_progress_bar=False),
                           dtype='float32').reshape(len(ns), 100)
            for j, n in enumerate(ns):
                if gold_pos[n]:
                    order = np.argsort(-s[j])
                    r[n] = min(int(np.where(order == p)[0][0]) for p in gold_pos[n])
            if a and a % (GROUP * 40) == 0:
                el = time.time() - t0
                print(f'    {a:,}/{len(ev):,}  {el:.0f}s '
                      f'(eta {el / a * (len(ev) - a) / 60:.1f} min)', flush=True)
        el = time.time() - t0
        rate[name] = len(ev) * 100 / el
        ranks[name] = r
        print(f'{name:<20} {rate[name]:>6,.0f} pairs/s | '
              f'R@1 {(r < 1).mean():>5.1%} R@5 {(r < 5).mean():>5.1%}  [{el / 60:.1f} min]',
              flush=True)
        del ce; gc.collect(); torch.cuda.empty_cache()
    except Exception as e:
        failed[name] = f'{type(e).__name__}: {e}'
        print(f'{name:<20} FAILED  {type(e).__name__}: {str(e)[:90]}')
        gc.collect(); torch.cuda.empty_cache()
print(f'\n{len(ranks)}/{len(MODELS)} candidates ran')

In [ ]:
res = {'bi-encoder': {'recall': rec(bi), 'mrr': mrr(bi), 'pairs_per_sec': None}}
for n in ranks:
    res[n] = {'recall': rec(ranks[n]), 'mrr': mrr(ranks[n]),
              'pairs_per_sec': float(rate[n])}

print(f"{'candidate':<22}" + ''.join(f'{("R@" + str(k)):>8}' for k in KS)
      + f"{'MRR':>8}{'pairs/s':>10}")
for n in ['bi-encoder'] + [m for _, m in [(0, x) for x in ranks]]:
    if n not in res:
        continue
    v = res[n]
    ps = f"{v['pairs_per_sec']:,.0f}" if v['pairs_per_sec'] else '-'
    print(f'{n:<22}' + ''.join(f"{v['recall'][k]:>8.1%}" for k in KS)
          + f"{v['mrr']:>8.3f}{ps:>10}")
print(f"{'ceiling':<22}{CEIL:>8.1%}")

print('\nREPRODUCTION CHECK vs notebook 6 (same val questions - must match)')
worst = 0.0
for n, ref in REF_NB6.items():
    if n not in res:
        print(f'  {n:<20} not run')
        continue
    d = max(abs(res[n]['recall'][k] - ref[k]) * 100 for k in ref)
    worst = max(worst, d)
    print(f'  {n:<20} ' + '  '.join(f"R@{k} {res[n]['recall'][k]:.1%} vs {ref[k]:.1%}"
                                    for k in (1, 5))
          + f'   max diff {d:.1f} pts')
print(f"  -> {'MATCH' if worst < 0.6 else 'MISMATCH - checkpoint or data path is wrong'}")

print('\nTHE DECISION, on one sample')
if 'bge-v2-m3' in res and 'minilm-finetuned' in res:
    a = res['minilm-finetuned']['recall'][5]
    b = res['bge-v2-m3']['recall'][5]
    fast = res['minilm-finetuned']['pairs_per_sec'] / res['bge-v2-m3']['pairs_per_sec']
    print(f'  fine-tuned MiniLM R@5 {a:.1%} vs bge-v2-m3 zero-shot R@5 {b:.1%} '
          f'({(a - b) * 100:+.1f} pts), MiniLM {fast:.0f}x faster')
    if a >= b:
        print('  -> SHIP fine-tuned MiniLM: at least as good AND far cheaper.')
        print('     Notebook 6\'s "3.9% short" verdict compared across samples and was wrong.')
    elif (b - a) * 100 < 2:
        print('  -> within ~2 pts; take MiniLM for the speed unless quality is critical.')
    else:
        print(f'  -> v2-m3 genuinely leads by {(b - a) * 100:.1f} pts. Next: fine-tune '
              'v2-m3 (~5-6 h, ~$3.50) or distil its rankings into MiniLM.')

print('\nPER-TYPE R@5 (the point of stratifying the split)')
vt = qtype[ev]
names = ['bi-encoder'] + list(ranks)
print(f"{'type':<10}{'n':>7}" + ''.join(f'{n[:16]:>18}' for n in names))
for t in sorted(set(vt.tolist())):
    m = vt == t
    if m.sum() < 20:
        continue
    cells = ''.join(f"{float(((bi if n == 'bi-encoder' else ranks[n])[m] < 5).mean()):>18.1%}"
                    for n in names)
    print(f'{t:<10}{int(m.sum()):>7,}{cells}')

In [ ]:
# §5 SAVE
for n in ranks:
    np.save(OUT / f'val_rank_{n}.npy', ranks[n])
np.save(OUT / 'val_rank_bi-encoder.npy', bi)
json.dump({'n_val': int(len(ev)), 'ceiling': CEIL,
           'models': {n: m for n, m in MODELS}, 'failed': failed,
           'results': {n: {'recall': {str(k): v for k, v in res[n]['recall'].items()},
                           'mrr': res[n]['mrr'],
                           'pairs_per_sec': res[n]['pairs_per_sec']} for n in res}},
          open(OUT / 'rerank_val.json', 'w'), indent=1)
print(f"-> {OUT / 'rerank_val.json'}")
nqrun.hf_push(str(OUT / 'rerank_val.json'), 'step4-rerank/rerank_val.json',
              f'rerankers on the {len(ev)}-question val split')
nqrun.log_metrics('7_rerank_val', n_val=int(len(ev)), ceiling=CEIL,
                  ran=','.join(ranks), failed=','.join(failed),
                  **{f'{n}_r@{k}'.replace('-', '_'): res[n]['recall'][k]
                     for n in res for k in KS},
                  **{f'{n}_pairs_per_sec'.replace('-', '_'): res[n]['pairs_per_sec']
                     for n in ranks})